# L3: Named entity recognition

Named entity recognition (NER) is the task of detecting and classifying _named entities_, such as persons or organizations.  It is a classic sequence labelling task where we want to annotate _spans_ within a text; i.e., sequences of one or more tokens.

In this lab, we will be using the [HuggingFace transformers](https://huggingface.co/docs/transformers/index) library, a popular and powerful Python library for working with some of the latest machine learning models.  This is a good library to be familiar with, as it allows you to use all kinds of models that are uploaded to the [HuggingFace Model Hub](https://huggingface.co/models), where many (if not most) of the state-of-the-art NLP models can be found.  As in the other labs, we don’t expect you to be familiar with this library yet, and will show examples and explanations for how to use it.

In [2]:
from liu_lt import check_solutions, find_file
from IPython.display import display
from pathlib import Path

## Loading and using an NER model

The model that we’ll be using is a BERT model that was fine-tuned on the NER task.  The following cell loads the model.  Note the following:

+ We’re creating a [“pipeline” for token classification](https://huggingface.co/docs/transformers/main_classes/pipelines), which is the most convenient way to use a HuggingFace model for NER.  “Token classification” here means the same as “sequence labelling”.
+ The model we are using is [dslim/bert-base-NER](https://huggingface.co/dslim/bert-base-NER).  More information about this model can be found on the HuggingFace Model Hub under the URL https://huggingface.co/dslim/bert-base-NER – the last part of the URL is always identical to the model name.
+ The HuggingFace library will emit some warnings, which you can ignore.

<div class="alert alert-warning">

⚠️ **If you run this on your own computer,** the first time you run this cell, it will automatically **download the model (ca.&nbsp;450&nbsp;MB)**.  This means you will need an internet connection and a little patience.

</div>

In [3]:
from transformers import pipeline

# This part checks if we’re on the lab computers, and loads the model from there
model = Path("/courses/729G86/models/dslim/bert-base-NER")
if not model.exists():
    model = "dslim/bert-base-NER"

# This instantiates the model pipeline
ner = pipeline("token-classification", model=model, aggregation_strategy="max")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


We can use this classification pipeline by simply calling it with a string as its argument:

In [4]:
ner("Born and raised in Paris, Brigitte Bardot was an aspiring ballerina during her childhood.")

[{'entity_group': 'LOC',
  'score': np.float32(0.9995901),
  'word': 'Paris',
  'start': 19,
  'end': 24},
 {'entity_group': 'PER',
  'score': np.float32(0.9983927),
  'word': 'Brigitte Bardot',
  'start': 26,
  'end': 41}]

You should see that calling `ner()` on some text returns a **list** of entities, where each entity is a **dictionary** as follows:

+ `entity_group` contains the label for the predicted entity.  The model distinguishes four **entity labels**:
  + **LOC**: a location
  + **ORG**: an organization
  + **PER**: a person's name
  + **MISC**: miscellaneous; all other entity types, such as names of events, nationalities, etc.
+ `score` is the probability score that the model predicted for this entity label; we will not do anything with this.
+ `word` is the string corresponding to the predicted entity (which can actually be multiple words, as it is a _span_ of text).
+ `start` is the **index** where the predicted entity **begins**.
+ `end` is the **index** where the predicted entity **ends**.

Try it out with some of your own examples before you continue! 👇

In [5]:
ner("Linköping University is a great place to study Language Technology in Sweden.")

[{'entity_group': 'ORG',
  'score': np.float32(0.9912553),
  'word': 'Linköping University',
  'start': 0,
  'end': 20},
 {'entity_group': 'MISC',
  'score': np.float32(0.70605844),
  'word': 'Language Technology',
  'start': 47,
  'end': 66},
 {'entity_group': 'LOC',
  'score': np.float32(0.99962413),
  'word': 'Sweden',
  'start': 70,
  'end': 76}]

## Loading the data

The data used in this lab consists of 1,000 English sentences that were annotated with named entities by the [UniversalNER](https://github.com/UniversalNER/UNER_English-PUD) project.  We already preprocessed the data to be in the same format that the transformer model’s pipeline returns, so that it is easier for you to work with.  The following cell loads the data and shows the first sentence:

In [6]:
import json

with open(find_file("ner-en-pud-test.json"), "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} sentences.")
print("Example:")
display(data[0])

Loaded 1000 sentences.
Example:


{'text': '“While much of the digital transition is unprecedented in the United States, the peaceful transition of power is not,” Obama special assistant Kori Schulman wrote in a blog post Monday.',
 'ents': [{'entity_group': 'LOC',
   'word': 'United States',
   'start': 62,
   'end': 75},
  {'entity_group': 'ORG', 'word': 'Obama', 'start': 119, 'end': 124},
  {'entity_group': 'PER', 'word': 'Kori Schulman', 'start': 143, 'end': 156}]}

You should see that each example from the dataset is a dictionary, where

+ `text` contains the sentence as a raw text string; and
+ `ents` contains a list of entities in the same format as above (but without a “score”).

The course module also provides a function that allows you to visualize an example:

In [7]:
from liu_lt.ner import display_ner

display_ner(data[0]["text"], data[0]["ents"])

## Task 1: Run the NER model on the dataset

Given the NER model and some test data, we can run the NER model on the data and then evaluate and analyse its performance.  As a first step, we need to run the NER model on the entire dataset.  Since this can take a bit longer to run, we do this once here at the beginning.

Since each example from the dataset is a dictionary (with the keys `text` and `ents`, as shown above), we want to do the following:

1. Iterate over each example in the dataset.
2. Run the NER model on the example.
3. Store the predicted entities under a new dictionary key `pred`.

<div class="alert alert-info">

  **Complete the code cell below** according to these instructions!  Note:

  - We have wrapped the iteration over the data in a call to `tqdm()`.  [tqdm](https://tqdm.github.io/) is a library that shows you a progress bar during the iteration, which can be helpful if your code runs a bit longer, so that you can see how it long it will take to finish.  It does not change anything about the values in `data`.
</div>

In [8]:
from tqdm import tqdm

for example in tqdm(data):
    # YOUR CODE HERE
    example["pred"] = ner(example["text"])

100%|██████████| 1000/1000 [00:16<00:00, 62.05it/s]


### 🤞 Test your code

If you did this correctly, each example should now also have a key `pred` with the predicted entities. The cell below checks if that appears to be the case:

In [9]:
with check_solutions() as check:
    check(
        all(example.keys() == {'text', 'ents', 'pred'} for example in data),
        "All examples in the data should now have 'text', 'ents', and 'pred' keys."
    )
    check(
        all(isinstance(example['pred'], list) for example in data),
        "All 'pred' keys should be lists with predicted entities."
    )

## Task 2: Qualitative analysis

When you run the following cell, you should see the first sentence from the dataset, first with the “true” annotated entities, and a second time with the “predicted” entities (from Task&nbsp;1).  Above the sentences, you should see a navigation bar where you can click to navigate between the sentences.  You can also enter a number in the input field, e.g., if you type “500” and press Enter you will see the 500th sentence from the dataset.

You should use this cell to **manually compare the “true” and “predicted” entity spans** by clicking through the sentences.  You are of course _not_ expected to look at _all_ 1000 sentences, but you should use this interface to get an idea for how often “true” and “predicted” match, and what some of the differences between “true” and “predicted” are, so that you can then write about your observations.

In [10]:
from liu_lt.ner import interactive_ner_browser

interactive_ner_browser(data)

<div class="alert alert-info">
  <strong>Write a brief relection describing your observations.</strong>  Use the following prompts:

  - What are some common differences that you found between spans in “true” and “predicted”?
  - Are there examples where “true” and “predicted” agree on the span, but not on the entity label?
  - How do you interpret these differences?  For example, is it because the model is wrong, the data is wrong, or something else?  (Remember that there is no universal definition for what a “correct” entity span is, and that, even though we call the annotation from the dataset the “true” entities, it might not be 100% accurate!)

  Always refer to concrete examples (e.g., “In sentences 503, 508, 510 we saw that...”).
</div>

Over prediction in example 268 where the model incorrectly tags "and Select" as entity spans when they are just adjectives describing "tariff". There is also span fragmentation within what should be a single phrase "Simple and Select" the model splits it into seperate entities with inconsistent labels ORG and MISC.

Label mismatches, model couldn't distinguish between ORG and MISC when the entity name does not clearly signal its category in example 167 "Liberal Democrats".

False positives for capitalised words - like "Yazidi" in example 25.

Model also struggles with multi-word organisation names, word boundary issues cause label inconsistency within what should be a single entity, like "Census Bureau" in example 18.

Many errors come from ambiguous spans and tokenisation.

## Task 3: Implement the evaluation functions

From the lectures, you should know how to compute span-level precision and recall.  Your first task is to implement functions that compute these metrics for a single sentence, based on the span representations that you saw above.

<div class="alert alert-info">
    <strong>Write code for each of the evaluation functions in the cells below!</strong>  Some hints:

  - A predicted entity is considered to be “correct” if the exact same entity span with the same entity label is found in the list of “true” entities.
  - Compare the `start` and `end` indices to determine if the span is correct, not the `word` – the same word could appear in different positions in a sentence!
  - If you need to divide by a number, make sure to check that number is not zero.  In each function, if you would divide by zero, the function should return zero instead.
</div>

### Span-level precision

This function takes a list of gold-standard entities (`true`) and a list of predicted entities (`pred`).  Each entity is represented as a dictionary in the way described at the top of this notebook.  The function should return the classifier’s precision as a floating-point number between 0 and 1.

In [ ]:
def precision(true, pred):
    """Compute the span-level precision of a classifier by comparing the true and predicted entities.
    
    Both `true` and `pred` are lists of dictionaries in the format described above.
    """
    # YOUR CODE HERE
    if len(pred) == 0:
        return 0
    
    correct_count = 0
    for p in pred:
        # check if predicted match true entity
        for t in true:
            if (p['start'] == t['start'] and 
                p['end'] == t['end'] and 
                p['entity_group'] == t['entity_group']):
                correct_count += 1
                break
    
    return correct_count / len(pred)


### Span-level recall

This function works the same as the previous one, but instead computes the recall.

In [ ]:
def recall(true, pred):
    """Compute the span-level recall of a classifier by comparing the true and predicted entities.
    
    Both `true` and `pred` are lists of dictionaries in the format described above.
    """
    # YOUR CODE HERE
    if len(true) == 0:
        return 0
    
    correct_count = 0
    for t in true:
        # check if this true match predicted 
        for p in pred:
            if (t['start'] == p['start'] and 
                t['end'] == p['end'] and 
                t['entity_group'] == p['entity_group']):
                correct_count += 1
                break
    
    return correct_count / len(true)

### 🤞 Test your code

Before you continue, you can test your implementation by running the following cell, which will compute some evaluation scores on a “toy” dataset and check if your functions return the expected values:

In [14]:
toy_pred = [
    {'entity_group': 'PER', 'word': 'Born', 'start': 0, 'end': 4},
    {'entity_group': 'LOC', 'word': 'United States', 'start': 62, 'end': 75},
    {'entity_group': 'PER', 'word': 'Obama', 'start': 89, 'end': 94},
    {'entity_group': 'ORG', 'word': 'Kori Schulman', 'start': 143, 'end': 156}
]
toy_true = [
    {'entity_group': 'LOC', 'word': 'United States', 'start': 62, 'end': 75},
    {'entity_group': 'PER', 'word': 'Obama', 'start': 119, 'end': 124},
    {'entity_group': 'ORG', 'word': 'Schulman', 'start': 148, 'end': 156}
]

with check_solutions() as check:
    check.isclose(precision(toy_true, toy_pred), 1/4)
    check.isclose(recall(toy_true, toy_pred), 1/3)
    check.isclose(precision(toy_true, toy_pred[:2]), 1/2)
    check.isclose(recall(toy_true[-2:], toy_pred[-2:]), 0)
    check.isclose(precision([], toy_pred), 0)
    check.isclose(recall(toy_true, []), 0)

## Task 4: Fine-grained evaluation on the full dataset

The implementation in Task 2 should have given you an idea how to compare the true and predicted entities, but it is not very useful yet for evaluating on the _entire_ dataset, for two reasons:

1. We can compute precision and recall **with respect to an entity label**, similar to how we have used these measures in text classification.
2. It's easier (and more accurate) to compute this **for all entities in the dataset** at once, rather than for a single sentence at a time.

We can implement this by first counting the number of true, predicted, and correct entity spans separately for each entity label.  To do this, we can use Python's [**Counter** object](https://docs.python.org/3/library/collections.html#counter-objects).  A “Counter” in Python works just like a dictionary that has integer values (_i.e._, counts), but with some added convenience.  For example, you do not need to initialize the dictionary with any values, but you can immediately start adding values for any key.  Here is an example:

In [15]:
from collections import Counter

c = Counter()
c["TEST"] += 1  # add 1 to the count of "TEST"
c["ABCD"] += 3  # add 3 to the count of "ABCD"
c["TEST"] += 1  # add 1 to the count of "TEST" (again)

c

Counter({'ABCD': 3, 'TEST': 2})

<div class="alert alert-info">

  **Complete the function below to count the number of true, predicted, and correct entity spans for each entity label!** Note the following:

  The function below already has a definition for the dictionary `label_counts`.  It has the following three keys:

  - `true` should contain the counts for the **true** entity spans.
  - `pred` should contain the counts for the **predicted** entity spans.
  - `correct` should contain the counts for how often the predicted spans were **correct**, i.e., were also in the true spans for the given example.

  Each of these contains a _Counter_ that you should use to count each entity label.  For example:
  - `label_counts["true"]["PER"]` should contain the value how often a "PER" span appeared in the gold-standard data.
  - `label_counts["pred"]["PER"]` should contain the value how often a "PER" span was predicted by the model.
  - `label_counts["correct"]["PER"]` should contain the value how often a "PER" span was _correctly_ predicted by the model.
  - ...and so on for all possibly entity types.
</div>

In [16]:
def evaluate_ner(data):
    """Count the number of true, predicted, and correct entity spans by entity label,
       and return the counts as a nested dictionary.
    """
    label_counts = {
        "true": Counter(),
        "pred": Counter(),
        "correct": Counter(),
    }

    # YOUR CODE HERE

    # iterate through each example 
    for example in data:
        true_ents = example["ents"]
        pred_ents = example["pred"]
        
        # count all true 
        for t in true_ents:
            label_counts["true"][t["entity_group"]] += 1
        
        # count all predicted
        for p in pred_ents:
            label_counts["pred"][p["entity_group"]] += 1
        
        # count correct predictions (matching span & label)
        for p in pred_ents:
            for t in true_ents:
                if (p['start'] == t['start'] and 
                    p['end'] == t['end'] and 
                    p['entity_group'] == t['entity_group']):
                    label_counts["correct"][p["entity_group"]] += 1
                    break
    
    return label_counts

### 🤞 Test your code

The cell below runs your evaluation function on the dataset that we created at the beginning, and checks some of the expected values:

In [17]:
counts = evaluate_ner(data)
display(counts)

with check_solutions() as check:
    check(counts["pred"].keys() == {'LOC', 'ORG', 'PER', 'MISC'})
    check.equals(counts["true"]["PER"], 414)
    check.equals(counts["pred"]["MISC"], 388)
    check.equals(counts["correct"]["LOC"], 373)

{'true': Counter({'LOC': 426, 'PER': 414, 'ORG': 235}),
 'pred': Counter({'LOC': 510, 'PER': 420, 'MISC': 388, 'ORG': 173}),
 'correct': Counter({'PER': 378, 'LOC': 373, 'ORG': 119})}

## Task 5: Interpret the quantitative evaluation

We are now ready to perform an evaluation on the entire dataset!  We can produce an _evaluation report_ similar to what we did for text classification by using the counts you produced in Task&nbsp;3.   The idea is as follows:

- **Precision** for an entity label X can be computed by dividing the number of _correct_ predictions for X by the number of _predicted entity spans_ with label X.
- **Recall** for an entity label Y can be computed by dividing the number of _correct_ predictions for X by the number of _true entity spans_ with label X.

This time, we provide you with a function that already does that for you and displays the results in a nice, tabular format.  Your task here is to **interpret the results** and what they mean.

In [18]:
from liu_lt.ner import evaluation_report

# Produce the evaluation report
counts = evaluate_ner(data)
evaluation_report(counts)

Entity label,Precision,Recall,F1-score
LOC,0.73,0.88,0.80
MISC,0.00,0.00,0.00
ORG,0.69,0.51,0.58
PER,0.90,0.91,0.91
Average,0.58,0.81,0.68


<div class="alert alert-info">
  <strong>Write a brief relection on the results from the evaluation report above.</strong>  Use the following prompts:

  - How do you interpret the results you got above?
  - What conclusions do you draw from them?  For example, is the model better at identifying some entity types than others?
  - Can you think of an easy way to improve the “average” precision and F1-score?
</div>

The model shows highly variable performance across entity types, with an overall F1-score of 0.68. 
The most striking is the dramatic difference between entity types: PER entities achieve excellent performance (F1=0.91, precision=0.90, recall=0.91), while MISC entities score 0.00 across all metrics, and ORG entities struggle significantly (F1=0.58).

Conclusions:
The model is very specialised, it performs well at identifying PER, having almost perfect precision and recall.
It fails at MISC entities and does not perform well on ORG entities. This implies that the model may be trained heavily on person names but lacks training for ambiguous/misc entity types. Recall is consistently higher than precision across most labels (except PER). This goes against the initial thought that the model is not conservative. It is over predicting entities broadly but failing to classify them correctly. The 0% MISC score tells us that the model never predicts MISC labels/all its MISC predictions are incorrect.

The simplest way to improve precision and F1-score would be to filter/re-calibrate MISC predictions. Since the model produces 0% accuracy on MISC entities, removing all MISC predictions and converting them to their most likely alternative labels could eliminate false positives without affecting overall performance negatively. Additionally, applying a confidence threshold to ORG predictions could reduce the 31% false positive rate (1 - 0.69 precision) on that label.

- - -

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

<strong>Before you submit...</strong>

➡️ Make sure you have filled in all cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.

➡️ **Make sure the notebook runs from start to finish** without errors, for example by clicking the "⏩" button in the toolbar, or via "Run$\rightarrow$Restart Kernel and Run All Cells..." in the menu.  If this is successful, you should see a green box appear below!

</div>

In [19]:
from liu_lt import success
success("Notebook runs from start to finish!")